In [ ]:
import os, sys

# Reliable hosted-Colab detection:
# - /content exists only in Google's managed VMs (not on your local machine)
# - TBE_CREDS_ADDR is set by Google's credential server, present only in hosted runtimes
IN_HOSTED_COLAB = os.path.exists('/content') and bool(os.environ.get('TBE_CREDS_ADDR'))

if IN_HOSTED_COLAB:
    !git clone https://github.com/jalengg/groundwork.git /content/groundwork
    %cd /content/groundwork
    !pip install -r requirements.txt -q
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/groundwork/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/groundwork/checkpoints', exist_ok=True)
    !ln -sf /content/drive/MyDrive/groundwork/data data
    !ln -sf /content/drive/MyDrive/groundwork/checkpoints checkpoints
else:
    # Local runtime — already running from the groundwork directory
    %cd /mnt/c/Users/jalen/groundwork

# Add project root to sys.path (for in-kernel imports) and PYTHONPATH
# (inherited by all !python subprocess calls)
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.environ['PYTHONPATH'] = project_root
print(f"Project root: {project_root}")

In [ ]:
# Run data pipeline for all cities. Takes ~2-3 hours total.
# Skip if data/ already populated from a previous session.
!python data_pipeline/cdg.py --config data_pipeline/cities.yaml --output data/

In [ ]:
!python model/train_vae.py \
    --data data/ \
    --output checkpoints/vae/ \
    --epochs 50 \
    --batch 4

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from model.vae import RoadVAE
from data_pipeline.dataset import RoadLayoutDataset

model = RoadVAE()
ckpt = torch.load('checkpoints/vae/vae_epoch_050.pth', map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()

ds = RoadLayoutDataset(['data/irving_tx'], augment=False)
_, road = ds[0]
with torch.no_grad():
    recon, _, _ = model(road.unsqueeze(0))
recon_argmax = recon[0].argmax(0).numpy()
road_argmax  = road.argmax(0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(road_argmax,  cmap='tab10', vmin=0, vmax=4); axes[0].set_title('Original')
axes[1].imshow(recon_argmax, cmap='tab10', vmin=0, vmax=4); axes[1].set_title('Reconstructed')
plt.show()
# Target: visually similar, SSIM > 0.70